<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex11.2-energy-optimization/Ex11.2_10_energy_optimization.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_11.2 — Energy Optimisation of Navigation with the Waveshare JetRacer

**Applied PINN for Energy · Aalborg University**

Runs **on the car**. Prerequisite: a working Ex_11.1 policy and its submission file.

---

### What you are being scored on

| Board | Metric | |
|---|---|---|
| **A — Efficiency** | `eta = J0 / J`, where `J = E_lap + w*T_lap` | `w = 5 J/s`, fixed for everyone |
| **B — Model fidelity** | `phi = E_meas / (E_meas + abs(E_pred - E_meas))` | prediction committed **before** the run |

Champion is the lowest rank sum across both boards; ties go to Board B.

### Gates
Three clean laps · the grip clamp binds at least once · lateral acceleration
never exceeds `mu*g` · predicted energy timestamped before the run.

## 0 · Identity, and the pack you actually have

The JetRacer Pro pack is **four 18650 cells at 8.4 V, wired two-in-series and
two-in-parallel**. The series pair sets the voltage; the parallel pair doubles
the capacity and roughly halves the internal resistance.

Verify it anyway. A tired pack is a different pack, and its resting voltage and
internal resistance both feed straight into your energy prediction.

**Note there is no motor encoder.** The Pro has none, so every speed in this
notebook comes from lap timing or from the camera. Plan for that before the
identification step — it is the single biggest practical constraint here.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex11.2-energy-optimization/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
TEAM = "team-01"
CAR  = 1

CELLS_IN_SERIES   = 2        # JetRacer Pro: 2S2P, four cells total
CELLS_IN_PARALLEL = 2
V_REST_MEASURED   = 8.3      # <- measure with a meter, car off, rested
CELL_CAPACITY_AH  = 3.0      # <- from your cell datasheet (per cell)
MASS_KG           = 1.05     # <- weigh your car, with pack
WHEELBASE_M       = 0.17     # <- measure it
MU_MEASURED       = 0.65     # <- measured today, on today's floor
COURSE_LENGTH_M   = 20.0
W_WEIGHT_J_PER_S  = 5.0      # FIXED by the instructor. Do not change.

import json, time, datetime, pathlib, math
import numpy as np

RUN_ID = f"{TEAM}_car{CAR:02d}"
DATA   = pathlib.Path(f"/home/jetson/ex11/{RUN_ID}")
(DATA / "logs").mkdir(parents=True, exist_ok=True)

PACK_CAPACITY_AH = CELL_CAPACITY_AH * CELLS_IN_PARALLEL      # parallel adds capacity
expected = 4.2 * CELLS_IN_SERIES                             # series sets voltage
if abs(V_REST_MEASURED - expected) > 0.8:
    print(f"WARNING: {CELLS_IN_SERIES}S should rest near {expected:.1f} V fully charged, "
          f"you measured {V_REST_MEASURED:.1f} V. Recharge, or you are modelling a tired pack.")
else:
    print(f"pack OK: {CELLS_IN_SERIES}S{CELLS_IN_PARALLEL}P, resting {V_REST_MEASURED:.1f} V")
print(f"Q = {PACK_CAPACITY_AH*3600:.0f} C,  m = {MASS_KG} kg,  L = {WHEELBASE_M} m,  mu = {MU_MEASURED}")
print("Reminder: 2P roughly halves R_int against a bare 2S string. Expect less sag "
      "than a series-only pack of the same voltage, and fit R_int rather than assuming it.")

## 1 · The INA219, configured deliberately

The motor drive is switched at a few kilohertz. Sampling at 20 Hz with a short
conversion window does not measure the average of that ripple — it measures
wherever in the PWM cycle each sample happened to land, and the trace is noise
dressed as data.

Set the conversion time and on-chip averaging so each reported sample integrates
over **many** switching periods, and record the configuration in your log header.
A trace whose sampling regime is unknown cannot be compared with anyone else's.

In [ ]:
from ina219 import INA219, DeviceRangeError

SHUNT_OHMS   = 0.1
MAX_EXPECTED_AMPS = 5.0

ina = INA219(SHUNT_OHMS, MAX_EXPECTED_AMPS)
# 12-bit resolution with 128-sample on-chip averaging: each reading integrates
# over ~68 ms, i.e. hundreds of PWM periods. This is the point of the cell.
ina.configure(voltage_range=ina.RANGE_16V,
              gain=ina.GAIN_AUTO,
              bus_adc=ina.ADC_128SAMP,
              shunt_adc=ina.ADC_128SAMP)

INA_CONFIG = {"shunt_ohms": SHUNT_OHMS, "max_amps": MAX_EXPECTED_AMPS,
              "bus_adc": "ADC_128SAMP", "shunt_adc": "ADC_128SAMP"}

def sample_power():
    """Returns (V_bus, I_amps, P_watts). Current is signed; discharge is positive."""
    v = ina.voltage()
    try:
        i = ina.current() / 1000.0          # library reports mA
    except DeviceRangeError:
        i = float("nan")
    return v, i, v * i

for _ in range(5):
    v, i, p = sample_power()
    print(f"{v:6.2f} V   {i:6.3f} A   {p:6.2f} W")
    time.sleep(0.2)

### Measure the hotel load

Car powered, policy **not** running, wheels off the ground. This is the constant
draw of the Nano, camera and OLED — the term that puts the interior minimum in
the energy-versus-lap-time curve. If it is invisible to your shunt, your shunt
is in the wrong place: it must see the whole car, not just the traction branch.

In [ ]:
samples = []
t0 = time.monotonic()
while time.monotonic() - t0 < 20.0:
    samples.append(sample_power()[2])
    time.sleep(0.05)

P_HOTEL_W = float(np.median(samples))
print(f"hotel load = {P_HOTEL_W:.2f} W  (median of {len(samples)} samples)")
print(f"over a 25 s lap that is {P_HOTEL_W*25:.0f} J before the car has moved a metre")
json.dump({"hotel_load_w": P_HOTEL_W, "ina": INA_CONFIG},
          open(DATA / "logs" / "hotel.json", "w"), indent=2)

## 2 · The instrumented loop

Your Ex_11.1 loop, with three boxes inserted: **constrain**, **sample**,
**buffer**. Steps 5 and 6 must be cheap or they will undo the loop rate you
fought for last week.

In [ ]:
from collections import deque

def grip_ceiling(steer_cmd, mu=MU_MEASURED, L=WHEELBASE_M, delta_max=math.radians(30.0)):
    """Hard layer: the maximum speed this steering angle physically permits.

    delta = delta_max * steer_cmd    (after calibration)
    R     = L / tan(delta)
    v_max = sqrt(mu * g * R)
    """
    delta = delta_max * max(-1.0, min(1.0, steer_cmd))
    if abs(delta) < 1e-4:
        return float("inf")
    R = L / abs(math.tan(delta))
    return math.sqrt(mu * 9.81 * R)

def clamp_throttle(throttle_cmd, v_now, steer_cmd, v_at_full_throttle=2.2):
    """Return (throttle, binding) with the grip limit enforced by construction."""
    v_max = grip_ceiling(steer_cmd)
    v_cmd = throttle_cmd * v_at_full_throttle
    if v_cmd <= v_max:
        return throttle_cmd, False
    return v_max / v_at_full_throttle, True

# quick sanity check, off the car
for s in (0.0, 0.25, 0.5, 1.0):
    print(f"steer {s:4.2f} -> R = {WHEELBASE_M/max(math.tan(math.radians(30)*s),1e-9):5.2f} m, "
          f"v_max = {grip_ceiling(s):5.2f} m/s")

In [ ]:
LOG = []

def drive_and_log(duration_s, policy_throttle=0.35, use_clamp=True):
    """policy from Ex_11.1 + grip clamp + INA219 sampling, all in one loop."""
    stop(); LOG.clear()
    t_start = time.monotonic()
    last_frame_t = t_start
    v_est = 0.0
    try:
        while time.monotonic() - t_start < duration_s:
            t0 = time.monotonic()
            frame = camera.value
            if frame is None or (t0 - last_frame_t) > STALE_S:
                stop(); continue                          # 1-2 capture, failsafe
            last_frame_t = t0
            with torch.no_grad():                          # 2 infer
                x, y = model(preprocess(frame))[0].tolist()
            steer = max(-1.0, min(1.0, x * STEER_GAIN + STEER_BIAS))
            thr, binding = (clamp_throttle(policy_throttle, v_est, steer)  # 3 constrain
                            if use_clamp else (policy_throttle, False))
            car.steering, car.throttle = steer, thr        # 4 actuate
            v, i, p = sample_power()                       # 5 sample
            LOG.append({"t": t0 - t_start, "dt": time.monotonic() - t0,   # 6 buffer
                        "V": v, "I": i, "P": p, "steer": steer,
                        "throttle": thr, "clamp": int(binding),
                        "v_max": grip_ceiling(steer)})
    finally:
        stop()
    return LOG

def energy_J(log):
    """Trapezoidal integration over irregular timestamps. Do not assume fixed dt."""
    t = np.array([e["t"] for e in log]); P = np.array([e["P"] for e in log])
    return float(np.trapz(P, t)) if len(t) > 1 else 0.0

## 3 · The coast-down

The single most informative manoeuvre on this platform. With the drive cut,
`i = 0`, every electrical term vanishes, and rolling resistance falls out of a
two-parameter fit almost by itself.

Bring the car to a steady speed on a straight, cut the drive, and log the
deceleration.

In [ ]:
def coast_down(spin_up_s=3.0, coast_s=6.0, throttle=0.45):
    """Log a coast-down. Speed is estimated from wheel-free deceleration; if you
    have an encoder or optical flow, substitute it here — it is worth the effort."""
    rec = []
    stop()
    car.throttle = throttle
    time.sleep(spin_up_s)
    car.throttle = 0.0
    t0 = time.monotonic()
    while time.monotonic() - t0 < coast_s:
        v, i, p = sample_power()
        rec.append({"t": time.monotonic() - t0, "V": v, "I": i, "P": p})
        time.sleep(0.02)
    stop()
    return rec

coast = coast_down()
json.dump(coast, open(DATA / "logs" / "coastdown.json", "w"))
print(f"{len(coast)} samples; residual draw during coast = "
      f"{np.median([r['P'] for r in coast]):.2f} W  (this should be your hotel load)")

## 4 · Parameter identification — the inverse problem from L7

Four coefficients are unknown and the model structure is known. That is exactly
the L7 inverse setting: promote the unknowns to trainable variables and minimise
a residual against measured data.

Longitudinal model:

    m * dv/dt = K * i  -  C_rr * m * g  -  0.5 * rho * Cd * A * v^2

where `K` is the **lumped traction constant** covering the RC380 motor drive,
the final drive and the wheel radius together. Do not try to separate them —
the data cannot distinguish them, and an optimiser handed both will happily
return a confident, meaningless pair.

The Pro is **4WD through front and rear differentials**. For a single-track
longitudinal model this changes nothing structural: the diffs and driveshafts
add drag that the identification absorbs into `C_rr`, which is why that
coefficient describes tyre, floor *and* transmission rather than tyre alone.

Electrical model:

    V_term = OCV(z) - i * R_int        and       P_elec = i^2 * R_a + tau * omega

**Scale before you optimise.** `R_a` is of order one ohm and `Q` of order ten
thousand coulombs; an optimiser handed both at once makes no progress on one.

In [ ]:
from scipy.optimize import least_squares

RHO, CD_A, G_ACC = 1.2, 0.012, 9.81      # air density, Cd*A, gravity

def simulate(params_scaled, t, i_meas, v0):
    """Forward-integrate the longitudinal model with scaled parameters."""
    C_rr, K, = params_scaled[0] * 1e-2, params_scaled[1] * 1e-1
    v = np.empty_like(t); v[0] = v0
    for k in range(1, len(t)):
        dt = t[k] - t[k-1]
        drag = 0.5 * RHO * CD_A * v[k-1] ** 2
        acc = (K * i_meas[k-1] - C_rr * MASS_KG * G_ACC - drag) / MASS_KG
        v[k] = max(0.0, v[k-1] + acc * dt)
    return v

def residual(params_scaled, t, i_meas, v_meas):
    return simulate(params_scaled, t, i_meas, v_meas[0]) - v_meas

# ── EXCITATION RUN: steps, ramps, a coast-down and a steady cruise ──
# A policy that only ever cruises leaves C_rr and R_a unidentifiable, because
# nothing in the data distinguishes them. Drive richly.
#
# THE CONSTRAINT: there is no encoder on this car. Your options, in order of
# effort and of quality:
#   (a) segment timing  — gates or floor marks at known spacing; coarse but honest
#   (b) optical flow    — from the camera you already have; noisy, free, workable
#   (c) an added sensor — a wheel encoder or an IMU, if your bench has one
# Whichever you choose, state it in the report: it bounds everything downstream.
#
#   excitation = drive_and_log(45.0, use_clamp=True)
#   t      = np.array([e["t"] for e in excitation])
#   i_meas = np.array([e["I"] for e in excitation])
#   v_meas = <your measured speed trace>      # see (a), (b) or (c) above
#
# fit = least_squares(residual, x0=[2.0, 1.0], args=(t, i_meas, v_meas),
#                     bounds=([0.1, 0.01], [20.0, 50.0]))
# C_rr, K = fit.x[0] * 1e-2, fit.x[1] * 1e-1
# print(f"C_rr = {C_rr:.4f}   K = {K:.3f} N/A")

print("Fill in your excitation run above. Sanity bounds when you do:")
print("  C_rr  a few hundredths        (0.01 - 0.05)")
print("  R_a   a few ohms              (0.5  - 5.0)")
print("  Negative values are not a numerical wobble. They mean the model is")
print("  absorbing something you have not included -- usually an uncalibrated")
print("  steering map or a units error in the gearing.")

### Do you believe the fit?

Four checks, all cheap, all worth more than a low residual:

1. **Hold out a run.** Fit on one drive, predict another.
2. **Check the signs.** A negative `C_rr` means the model is absorbing an error.
3. **Compare magnitudes.** Values orders away from the ranges above indicate a units error.
4. **Refit warm and cold.** Winding resistance rises as the motors heat. If the
   two fits differ, that is physics, and it belongs in your report.

## 5 · The energy-optimal speed profile

Minimise `J = E + w*T` subject to the grip limit at every point of the course.

Two results from the lecture drive the answer, and neither is obvious:

- **Winding loss is quadratic in current**, so for a fixed mean tractive force
  the mean square is smallest when the force is *constant*. Jensen's inequality,
  not a heuristic. Surging is paid for twice.
- **There is no regeneration.** Kinetic energy shed before an apex becomes heat.
  Braking late is fast in racing, where energy is free. Here it is not.

So the cheapest lap is the **flattest** speed profile that still respects the
grip limit at every corner — not the slowest.

In [ ]:
V_MAX_CAR = 1.8      # <- MEASURE: top speed at your capped throttle, m/s

def optimal_profile(curvature, ds, mu=MU_MEASURED, w=W_WEIGHT_J_PER_S,
                    P_hotel=None, K=1.0, R_a=1.5, C_rr=0.02,
                    v_max_car=None):
    """Discretised speed profile minimising E + w*T over one lap.

    curvature : array of 1/R at each segment (1/m), from your course survey
    ds        : segment length (m)
    """
    P_hotel = P_hotel if P_hotel is not None else P_HOTEL_W
    v_cap = v_max_car if v_max_car is not None else V_MAX_CAR
    # per-segment ceiling: the tighter of the grip limit and the car's own top speed
    v_lim = np.array([min(math.sqrt(mu * 9.81 / k), v_cap) if k > 1e-6 else v_cap
                      for k in curvature])

    def cost(v):
        v = np.clip(v, 0.2, v_lim)                  # grip enforced by construction
        dt = ds / v
        F = C_rr * MASS_KG * 9.81 + MASS_KG * np.gradient(v) * v / ds
        i = np.maximum(F / max(K, 1e-6), 0.0)
        P = i ** 2 * R_a + F * v + P_hotel
        return float(np.sum(P * dt) + w * np.sum(dt))

    from scipy.optimize import minimize
    v0 = np.minimum(v_lim, 1.5)
    res = minimize(cost, v0, method="L-BFGS-B",
                   bounds=[(0.2, float(u)) for u in v_lim])
    v_opt = np.clip(res.x, 0.2, v_lim)
    T = float(np.sum(ds / v_opt))
    E = float(cost(v_opt) - w * T)
    return v_opt, v_lim, E, T

# ── your course survey: curvature at each 0.5 m segment ──
# PLACEHOLDER. Replace with your own measurement: pace the course, note the
# radius of each corner, and write 1/R for the segments inside it. The numbers
# below describe a 20 m lap with three corners and mean nothing about yours.
ds = 0.5
curvature = np.array([0.0]*8 + [2.0]*6 + [0.0]*10 + [3.3]*4 + [0.0]*6 + [1.4]*6)
v_opt, v_lim, E_pred, T_pred = optimal_profile(curvature, ds)

J_pred = E_pred + W_WEIGHT_J_PER_S * T_pred
print(f"predicted lap time   {T_pred:6.2f} s")
print(f"predicted energy     {E_pred:6.1f} J")
print(f"predicted cost J     {J_pred:6.1f} J     (w = {W_WEIGHT_J_PER_S} J/s)")
print(f"speed range          {v_opt.min():.2f} - {v_opt.max():.2f} m/s")

### Commit the prediction (gate)

Write it down **before** you drive. This cell timestamps it. Board B compares
this file against the INA219 trace, and a prediction produced after the run is
not a prediction.

In [ ]:
commitment = {
    "team": TEAM, "car": CAR,
    "E_pred_J": E_pred, "T_pred_s": T_pred, "J_pred": J_pred,
    "w_J_per_s": W_WEIGHT_J_PER_S,
    "hotel_load_w": P_HOTEL_W,
    "mu_measured": MU_MEASURED,
    "speed_profile_ms": [round(float(x), 3) for x in v_opt],
    "committed_at": datetime.datetime.now().isoformat(timespec="seconds"),
}
path = DATA / f"COMMITMENT_Ex11.2_{RUN_ID}.json"
if path.exists():
    print(f"REFUSING to overwrite an existing commitment at {path}")
else:
    path.write_text(json.dumps(commitment, indent=2))
    print(f"committed {E_pred:.1f} J at {commitment['committed_at']}")

## 6 · The scored run, and the two indices

Three clean laps driving your optimal profile. The clamp must bind at least once
and must never be violated — otherwise the profile was never near the limit and
the exercise was not attempted.

In [ ]:
# ── baseline: your car's stock fixed-throttle run, done before the session ──
E0_BASELINE_J = 142.0
T0_BASELINE_S = 24.8
# ────────────────────────────────────────────────────────────────────────────

run = drive_and_log(duration_s=90.0, use_clamp=True)
json.dump(run, open(DATA / "logs" / "scored_run.json", "w"))

E_meas = energy_J(run)
LAP_TIMES_S = [21.9, 22.1, 21.7]     # from the marshal
CONTACTS, INTERVENTIONS = 0, 0

import statistics
assert INTERVENTIONS == 0, "run voided by manual intervention"
T_meas = statistics.median(LAP_TIMES_S)
E_lap  = E_meas * (T_meas / (run[-1]["t"] or 1.0))     # energy attributed to one lap

clamp_bound = sum(e["clamp"] for e in run) > 0
violations  = sum(1 for e in run if e["clamp"] == 0 and
                  e["throttle"] * 2.2 > e["v_max"] + 1e-6)

J0   = E0_BASELINE_J + W_WEIGHT_J_PER_S * T0_BASELINE_S
J    = E_lap + W_WEIGHT_J_PER_S * T_meas
eta  = J0 / J
phi  = E_lap / (E_lap + abs(E_pred - E_lap)) if E_lap > 0 else 0.0

gates = {"three_clean_laps": len(LAP_TIMES_S) == 3 and CONTACTS == 0,
         "clamp_binds": bool(clamp_bound),
         "clamp_holds": violations == 0,
         "committed": path.exists()}

print(f"measured energy per lap  {E_lap:7.1f} J     (predicted {E_pred:.1f} J)")
print(f"measured lap time        {T_meas:7.2f} s     (predicted {T_pred:.2f} s)")
print(f"cost J                   {J:7.1f} J     baseline J0 {J0:.1f} J")
print(f"\nBOARD A  efficiency eta = {eta:.3f}")
print(f"BOARD B  fidelity   phi = {phi:.3f}")
print("\nGATES:")
for k, v in gates.items():
    print(f"  [{'OK ' if v else 'FAIL'}] {k}")

submission = dict(team=TEAM, car=CAR, E_lap_J=E_lap, T_lap_s=T_meas,
                  E_pred_J=E_pred, T_pred_s=T_pred,
                  E0_J=E0_BASELINE_J, T0_s=T0_BASELINE_S,
                  w_J_per_s=W_WEIGHT_J_PER_S, J=J, J0=J0,
                  eta=eta, phi=phi, gates=gates,
                  clamp_bind_fraction=sum(e["clamp"] for e in run) / len(run),
                  timestamp=datetime.datetime.now().isoformat(timespec="seconds"))
sp = DATA / f"SUBMISSION_Ex11.2_{RUN_ID}.json"
sp.write_text(json.dumps(submission, indent=2))
print(f"\nwrote {sp}")

## 7 · Hand in

- `SUBMISSION_Ex11.2_<team>_car<NN>.json`
- `COMMITMENT_Ex11.2_<team>_car<NN>.json`
- `logs/scored_run.json`, `logs/coastdown.json`, `logs/hotel.json`

And **account for the gap** between predicted and measured energy. A team that
predicts 118 J, measures 141 J, and explains the difference as warm windings has
done better physics than a team that predicts 118 J, measures 119 J, and cannot
say why.

Suspiciously perfect agreement usually means the prediction was tuned after the
measurement. Board B is ranked, but the report is marked.